# Download do Dataset LUNA16

Este notebook faz o download do dataset LUNA16 a partir do Zenodo e organiza
os arquivos na estrutura esperada pelo projeto.

O dataset esta dividido em duas partes:
- **Part 1** (https://zenodo.org/records/3723295): annotations.csv, candidates.csv e subset0 a subset6
- **Part 2** (https://zenodo.org/records/4121926): subset7 a subset9

Cada subset contem CT scans no formato `.mhd` + `.raw` (pares de arquivos).

O download suporta **resume** em caso de queda de conexao. Arquivos ja existentes
em `data/luna/` sao automaticamente ignorados.

In [1]:
import os
import zipfile
from pathlib import Path
import shutil

import requests
from tqdm import tqdm

## Configuracao

In [4]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
LUNA_DIR = PROJECT_ROOT / "data" / "luna"

print(f"PROJECT_ROOT: {PROJECT_ROOT}\nRAW_DIR: {RAW_DIR}\nLUNA_DIR: {LUNA_DIR}\n")

RAW_DIR.mkdir(parents=True, exist_ok=True)
LUNA_DIR.mkdir(parents=True, exist_ok=True)

ZENODO_PART1 = "https://zenodo.org/records/3723295/files"
ZENODO_PART2 = "https://zenodo.org/records/4121926/files"

# Mapeamento de todos os arquivos para download
LUNA16_FILES = {
    # CSVs (Part 1)
    "annotations.csv": f"{ZENODO_PART1}/annotations.csv?download=1",
    "candidates.csv": f"{ZENODO_PART1}/candidates.csv?download=1",
    # Subsets (Part 1)
    "subset0.zip": f"{ZENODO_PART1}/subset0.zip?download=1",
    "subset1.zip": f"{ZENODO_PART1}/subset1.zip?download=1",
    "subset2.zip": f"{ZENODO_PART1}/subset2.zip?download=1",
    "subset3.zip": f"{ZENODO_PART1}/subset3.zip?download=1",
    "subset4.zip": f"{ZENODO_PART1}/subset4.zip?download=1",
    "subset5.zip": f"{ZENODO_PART1}/subset5.zip?download=1",
    "subset6.zip": f"{ZENODO_PART1}/subset6.zip?download=1",
    # Subsets (Part 2)
    "subset7.zip": f"{ZENODO_PART2}/subset7.zip?download=1",
    "subset8.zip": f"{ZENODO_PART2}/subset8.zip?download=1",
    "subset9.zip": f"{ZENODO_PART2}/subset9.zip?download=1",
}


print(f"Total de arquivos: {len(LUNA16_FILES)}")

PROJECT_ROOT: /Users/carlos/Library/CloudStorage/Dropbox/Sigmoidal/Cursos/Deep Learning para Visão Computacional/bootcamp-deep-learning
RAW_DIR: /Users/carlos/Library/CloudStorage/Dropbox/Sigmoidal/Cursos/Deep Learning para Visão Computacional/bootcamp-deep-learning/data/raw
LUNA_DIR: /Users/carlos/Library/CloudStorage/Dropbox/Sigmoidal/Cursos/Deep Learning para Visão Computacional/bootcamp-deep-learning/data/luna

Total de arquivos: 12


## Funcoes auxiliares

In [5]:
def download_file(url, dest_path, chunk_size=8192):
    """Baixa um arquivo com suporte a resume.

    Se o arquivo ja existe parcialmente em dest_path, continua o download
    de onde parou. Se ja esta completo, pula.

    Args:
        url: URL do arquivo.
        dest_path: Caminho de destino (Path).
        chunk_size: Tamanho do bloco de leitura em bytes.

    Returns:
        True se o download foi realizado, False se o arquivo ja existia.
    """
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)

    # Verificar tamanho remoto
    head = requests.head(url, allow_redirects=True, timeout=30)
    remote_size = int(head.headers.get("Content-Length", 0))

    # Se arquivo ja existe com tamanho correto, pular
    if dest_path.exists():
        local_size = dest_path.stat().st_size
        if local_size >= remote_size and remote_size > 0:
            print(f"  [OK] {dest_path.name} ja existe ({local_size / 1e6:.1f} MB)")
            return False
    else:
        local_size = 0

    # Headers para resume
    headers = {}
    if local_size > 0:
        headers["Range"] = f"bytes={local_size}-"
        print(f"  Resumindo {dest_path.name} a partir de {local_size / 1e6:.1f} MB...")

    response = requests.get(url, headers=headers, stream=True, timeout=60)
    response.raise_for_status()

    # Calcular total para a barra de progresso
    content_length = int(response.headers.get("Content-Length", 0))
    total = content_length if content_length > 0 else (remote_size - local_size)

    mode = "ab" if local_size > 0 else "wb"

    with open(dest_path, mode) as f:
        with tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=dest_path.name,
            initial=0,
        ) as pbar:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    return True

In [ ]:
def extract_subset_zip(zip_path, luna_dir):
    """Extrai um subset zip para a estrutura correta em luna_dir.

    Os zips do LUNA16 criam uma pasta aninhada (ex: subset0/subset0/*.mhd).
    Esta funcao extrai e move os arquivos para luna_dir/subset0/*.mhd.

    Args:
        zip_path: Caminho do arquivo zip.
        luna_dir: Diretorio base do dataset (data/luna/).
    """
    zip_path = Path(zip_path)
    luna_dir = Path(luna_dir)
    subset_name = zip_path.stem  # ex: "subset0"
    target_dir = luna_dir / subset_name

    if target_dir.exists() and any(target_dir.glob("*.mhd")):
        print(f"  [OK] {subset_name} ja esta extraido ({len(list(target_dir.glob('*.mhd')))} scans)")
        return

    print(f"  Extraindo {zip_path.name}...")
    temp_dir = luna_dir / f"_temp_{subset_name}"

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_dir)

    # O zip cria subset0/subset0/*.mhd -- precisamos mover para subset0/*.mhd
    nested_dir = temp_dir / subset_name / subset_name
    if nested_dir.exists():
        source = nested_dir
    elif (temp_dir / subset_name).exists():
        source = temp_dir / subset_name
    else:
        source = temp_dir

    target_dir.mkdir(parents=True, exist_ok=True)
    for f in source.iterdir():
        shutil.move(str(f), str(target_dir / f.name))

    # Limpar temp
    shutil.rmtree(temp_dir, ignore_errors=True)

    mhd_count = len(list(target_dir.glob("*.mhd")))
    print(f"  [OK] {subset_name} extraido ({mhd_count} scans)")

In [6]:
def is_subset_ready(subset_name, luna_dir):
    """Verifica se um subset ja esta extraido em luna_dir."""
    target_dir = Path(luna_dir) / subset_name
    return target_dir.exists() and any(target_dir.glob("*.mhd"))


def download_and_extract(file_name, url, raw_dir, luna_dir):
    """Baixa e extrai um arquivo do LUNA16.

    Para CSVs, copia direto para luna_dir.
    Para ZIPs, baixa em raw_dir e extrai para luna_dir.
    Se o conteudo ja existe em luna_dir, pula o download.
    """
    raw_dir = Path(raw_dir)
    luna_dir = Path(luna_dir)

    if file_name.endswith(".csv"):
        dest = luna_dir / file_name
        if dest.exists() and dest.stat().st_size > 0:
            print(f"  [OK] {file_name} ja existe em luna/")
            return
        download_file(url, dest)
        return

    if file_name.endswith(".zip"):
        subset_name = file_name.replace(".zip", "")

        # Se o subset ja esta extraido, nao baixar o zip
        if is_subset_ready(subset_name, luna_dir):
            print(f"  [OK] {subset_name} ja existe em luna/ -- pulando download")
            return

        zip_path = raw_dir / file_name
        download_file(url, zip_path)
        extract_subset_zip(zip_path, luna_dir)
        return

## Download de um subconjunto (teste)

Antes de baixar o dataset inteiro, teste com um unico subset para validar
que o download e a extracao estao funcionando corretamente.

In [8]:
# Teste com subset0 e os CSVs
test_files = ["annotations.csv", "candidates.csv", "subset0.zip"]

for file_name in test_files:
    print(f"\nProcessando: {file_name}")
    url = LUNA16_FILES[file_name]
    download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)


Processando: annotations.csv
  [OK] annotations.csv ja existe em luna/

Processando: candidates.csv
  [OK] candidates.csv ja existe em luna/

Processando: subset0.zip
  Resumindo subset0.zip a partir de 581.9 MB...


subset0.zip:   2%|▏         | 113M/6.23G [00:12<11:36, 8.78MB/s]    


KeyboardInterrupt: 

## Verificacao rapida

In [9]:
# Verificar o que temos em luna/
print("Conteudo de data/luna/:\n")
for item in sorted(LUNA_DIR.iterdir()):
    if item.is_dir():
        mhd_files = list(item.glob("*.mhd"))
        print(f"  {item.name}/  ({len(mhd_files)} scans)")
    else:
        print(f"  {item.name}  ({item.stat().st_size / 1e6:.1f} MB)")

Conteudo de data/luna/:

  .DS_Store  (0.0 MB)
  annotations.csv  (0.1 MB)
  candidates.csv  (55.4 MB)
  subset0/  (0 scans)
  subset1/  (0 scans)
  subset2/  (0 scans)
  subset3/  (0 scans)
  subset4/  (0 scans)
  subset5/  (0 scans)
  subset6/  (0 scans)
  subset7/  (0 scans)
  subset8/  (0 scans)
  subset9/  (0 scans)


## Download do dataset completo

Execute a celula abaixo para baixar todos os 10 subsets e os CSVs.
O download total e de aproximadamente 67 GB. Arquivos ja presentes em
`data/luna/` serao automaticamente ignorados.

In [10]:
for file_name, url in LUNA16_FILES.items():
    print(f"\nProcessando: {file_name}")
    try:
        download_and_extract(file_name, url, RAW_DIR, LUNA_DIR)
    except Exception as e:
        print(f"  [ERRO] {file_name}: {e}")
        print(f"  Re-execute esta celula para tentar novamente.")


Processando: annotations.csv
  [OK] annotations.csv ja existe em luna/

Processando: candidates.csv
  [OK] candidates.csv ja existe em luna/

Processando: subset0.zip
  Resumindo subset0.zip a partir de 694.6 MB...


subset0.zip:   3%|▎         | 167M/6.12G [00:32<19:16, 5.15MB/s]   


KeyboardInterrupt: 

## Verificacao final

In [ ]:
print("Status do dataset LUNA16:\n")

# CSVs
for csv_name in ["annotations.csv", "candidates.csv"]:
    csv_path = LUNA_DIR / csv_name
    status = "presente" if csv_path.exists() else "FALTANDO"
    print(f"  {csv_name}: {status}")

print()

# Subsets
total_scans = 0
for i in range(10):
    subset_dir = LUNA_DIR / f"subset{i}"
    if subset_dir.exists():
        mhd_count = len(list(subset_dir.glob("*.mhd")))
        total_scans += mhd_count
        print(f"  subset{i}: {mhd_count} scans")
    else:
        print(f"  subset{i}: FALTANDO")

print(f"\nTotal de CT scans: {total_scans}")